In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Extracao incremental -> Bronze
# MAGIC
# MAGIC Busca issues e PRs do GitHub **a partir do watermark de cada repositorio** e grava o JSON cru no Bronze (append-only).
# MAGIC
# MAGIC Decisoes de desenho:
# MAGIC - A API e consultada com `since=<watermark>`, `sort=updated`, `direction=asc`.
# MAGIC - O watermark novo e o maior `updated_at` **recebido** (nunca o relogio local) e so avanca **depois** do commit no Bronze.
# MAGIC - Ordem ascendente + limite de paginas = extracao retomavel: se truncar, a proxima execucao continua de onde parou.
# MAGIC - `since` e inclusivo, entao o ultimo item volta a ser buscado. Isso gera duplicata no Bronze, resolvida pelo MERGE no Silver.

# COMMAND ----------

# MAGIC %run ./00_common

# COMMAND ----------

import requests

dbutils.widgets.text("repos", "delta-io/delta,dbt-labs/dbt-core,great-expectations/great_expectations")
dbutils.widgets.text("backfill_days", "7")
dbutils.widgets.text("max_pages_per_repo", "10")
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"])

REPOS = [r.strip() for r in dbutils.widgets.get("repos").split(",") if r.strip()]
BACKFILL_DAYS = int(dbutils.widgets.get("backfill_days"))
MAX_PAGES = int(dbutils.widgets.get("max_pages_per_repo"))
FULL_REFRESH = dbutils.widgets.get("full_refresh") == "true"

# Token e opcional. Sem token: 60 req/h. Com token: 5000 req/h.
# Para usar: crie um secret scope "github" com a chave "token".
try:
    GITHUB_TOKEN = dbutils.secrets.get("github", "token")
except Exception:
    GITHUB_TOKEN = None

print(f"Repos: {REPOS}")
print(f"Backfill (primeira carga): {BACKFILL_DAYS} dias | max paginas/repo: {MAX_PAGES} | full_refresh: {FULL_REFRESH}")
print(f"Autenticado: {GITHUB_TOKEN is not None}")